# Seasonal Agriculture Performance Analysis

**Student Name:** Ranadeep Das  
**AICTE STU ID:** `STU6a5f732ab94b61784640298`

## Project Overview

This project analyzes agricultural data across different seasons to understand patterns in crop yield, profitability, environmental conditions, irrigation, water usage and farming-related factors.

The main focus is to compare **Kharif, Rabi and Zaid** seasons and identify useful patterns through data cleaning, exploratory analysis, grouped summaries and visualization.

### Tools Used
- Python
- Pandas
- NumPy
- Matplotlib
- Jupyter Notebook


## 1. Problem Statement

Agricultural performance can change from one season to another because of differences in environmental conditions, farming practices, resource availability and market conditions.

The aim of this analysis is to use the available agricultural data to identify meaningful seasonal differences in **yield, profit, resource usage and farming conditions**.


## 2. Objectives

1. Explore and understand the dataset.
2. Check and clean missing or duplicate data.
3. Compare agricultural performance across seasons.
4. Study crop-level performance.
5. Examine irrigation and water-efficiency patterns.
6. Investigate relationships between important numerical variables.
7. Create clear visualizations.
8. Summarize the major findings and provide practical recommendations.


## 3. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

print("Libraries imported successfully.")


## 4. Load the Dataset

The dataset is kept in the `data` folder of the GitHub project. The relative path below is therefore used so that the notebook can be run directly from the `notebooks` folder.


In [ ]:
df = pd.read_csv("../data/seasonal_agriculture_performance_dataset.csv")

print("Dataset loaded successfully.")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])


## 5. First Look at the Data

In [ ]:
df.head()


In [ ]:
df.tail()


### Dataset Shape

In [ ]:
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])


### Column Names

In [ ]:
print(df.columns.tolist())


### Data Types and Structure

In [ ]:
df.info()


## 6. Data Quality Check

Before doing the analysis, I checked for missing values and duplicate rows.


In [ ]:
missing_values = df.isnull().sum()
missing_values[missing_values > 0].sort_values(ascending=False)


In [ ]:
duplicate_count = df.duplicated().sum()
print("Duplicate rows:", duplicate_count)


### Missing-Value Summary

The supplied dataset contains missing values in:
- `Rainfall_mm`
- `Soil_Moisture_pct`
- `Yield_Tonnes_Ha`

For the numerical variables above, I used **median imputation**. This keeps all records while reducing the effect of unusually high or low observations.


In [ ]:
numeric_impute_cols = [
    "Rainfall_mm",
    "Soil_Moisture_pct",
    "Yield_Tonnes_Ha"
]

missing_before = df[numeric_impute_cols].isnull().sum()
print("Missing values before cleaning:")
print(missing_before)

for col in numeric_impute_cols:
    df[col] = df[col].fillna(df[col].median())

print("\nMissing values after cleaning:")
print(df[numeric_impute_cols].isnull().sum())


## 7. Basic Descriptive Statistics

In [ ]:
df.describe(include="all").T


## 8. Understand the Main Categories

In [ ]:
for col in ["Season", "Crop", "State", "Irrigation_Method"]:
    print(f"\n--- {col} ---")
    print(df[col].value_counts())


## 9. Seasonal Performance Analysis

The first major question is:

> **How does agricultural performance vary across seasons?**

I compare median yield and median profit because both variables can contain extreme observations.


In [ ]:
season_order = ["Kharif", "Rabi", "Zaid"]

season_summary = df.groupby("Season").agg(
    Farms=("Farm_ID", "count"),
    Median_Yield=("Yield_Tonnes_Ha", "median"),
    Median_Profit=("Profit_INR", "median"),
    Average_Rainfall=("Rainfall_mm", "mean"),
    Average_Soil_Moisture=("Soil_Moisture_pct", "mean"),
    Average_Water_Efficiency=("Water_Efficiency_t_per_1000m3", "mean"),
    Average_Pest_Risk=("Disease_Pest_Risk_pct", "mean")
).reindex(season_order)

season_summary


### 9.1 Median Yield by Season

In [ ]:
plt.figure(figsize=(8, 5))
plt.bar(season_order, season_summary["Median_Yield"])
plt.title("Median Agricultural Yield by Season")
plt.xlabel("Season")
plt.ylabel("Yield (tonnes/hectare)")
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()


### 9.2 Median Profit by Season

In [ ]:
plt.figure(figsize=(8, 5))
profit_lakh = season_summary["Median_Profit"] / 100000

plt.bar(season_order, profit_lakh)
plt.axhline(0, linewidth=1)
plt.title("Median Profit by Season")
plt.xlabel("Season")
plt.ylabel("Profit (₹ lakh)")
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()


### 9.3 Percentage of Profitable Farms

In [ ]:
profitable_rate = (
    df.groupby("Season")["Profit_INR"]
      .apply(lambda x: (x > 0).mean() * 100)
      .reindex(season_order)
)

plt.figure(figsize=(8, 5))
plt.bar(season_order, profitable_rate)
plt.title("Percentage of Farms Making a Positive Profit")
plt.xlabel("Season")
plt.ylabel("Profitable farms (%)")
plt.ylim(0, 70)
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

profitable_rate


### Seasonal Interpretation

From the grouped analysis, **Kharif shows the strongest overall performance** among the three seasons. It has the highest median yield and the highest median profit.

**Zaid shows the weakest profitability**, with a negative median profit in the supplied data.

These are descriptive observations from this dataset and should not be treated as proof that season alone causes the differences.


## 10. Seasonal Environmental Conditions

Next, I checked whether the seasons also differ in environmental conditions such as rainfall, soil moisture and disease/pest risk.


In [ ]:
environment_summary = df.groupby("Season").agg(
    Rainfall_mm=("Rainfall_mm", "mean"),
    Soil_Moisture_pct=("Soil_Moisture_pct", "mean"),
    Temperature_C=("Temperature_C", "mean"),
    Humidity_pct=("Humidity_pct", "mean"),
    Disease_Pest_Risk_pct=("Disease_Pest_Risk_pct", "mean")
).reindex(season_order)

environment_summary


### 10.1 Rainfall by Season

In [ ]:
plt.figure(figsize=(8, 5))
plt.bar(season_order, environment_summary["Rainfall_mm"])
plt.title("Average Rainfall by Season")
plt.xlabel("Season")
plt.ylabel("Rainfall (mm)")
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()


### 10.2 Water Efficiency by Season

In [ ]:
plt.figure(figsize=(8, 5))
plt.bar(season_order, season_summary["Average_Water_Efficiency"])
plt.title("Average Water Efficiency by Season")
plt.xlabel("Season")
plt.ylabel("Tonnes per 1,000 m³")
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()


### Interpretation

The environmental variables show a clear seasonal difference. Kharif is associated with substantially higher rainfall and soil moisture, while Zaid is comparatively drier.

This gives useful context for understanding why agricultural outcomes may differ across seasons.


## 11. Crop-Level Analysis

Another question is:

> **Which crops perform better, and does their performance change between seasons?**


In [ ]:
crop_summary = df.groupby("Crop").agg(
    Farms=("Farm_ID", "count"),
    Median_Yield=("Yield_Tonnes_Ha", "median"),
    Median_Profit=("Profit_INR", "median"),
    Average_Production=("Production_Tonnes", "mean")
).sort_values("Median_Yield", ascending=False)

crop_summary


### 11.1 Crop Yield Across Seasons

In [ ]:
crop_season_yield = (
    df.groupby(["Crop", "Season"])["Yield_Tonnes_Ha"]
      .median()
      .unstack()
      .reindex(columns=season_order)
)

crop_season_yield


In [ ]:
x = np.arange(len(crop_season_yield.index))
width = 0.25

plt.figure(figsize=(11, 6))

for i, season in enumerate(season_order):
    plt.bar(
        x + (i - 1) * width,
        crop_season_yield[season],
        width,
        label=season
    )

plt.xticks(x, crop_season_yield.index, rotation=25, ha="right")
plt.title("Median Crop Yield Across Seasons")
plt.xlabel("Crop")
plt.ylabel("Yield (tonnes/hectare)")
plt.legend()
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()


### 11.2 Median Profit by Crop

In [ ]:
crop_profit = crop_summary["Median_Profit"].sort_values(ascending=False) / 100000

plt.figure(figsize=(10, 5))
plt.bar(crop_profit.index, crop_profit)
plt.axhline(0, linewidth=1)
plt.title("Median Profit by Crop")
plt.xlabel("Crop")
plt.ylabel("Profit (₹ lakh)")
plt.xticks(rotation=25, ha="right")
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()


### Crop Interpretation

Sugarcane stands out as a high-yield and high-profit crop in this dataset. Chilli also shows comparatively strong profitability.

For many crops, the median yield is higher in Kharif than in the other seasons.


## 12. Irrigation Analysis

I also looked at whether irrigation method is associated with differences in yield and water efficiency.


In [ ]:
irrigation_summary = df.groupby("Irrigation_Method").agg(
    Farms=("Farm_ID", "count"),
    Median_Yield=("Yield_Tonnes_Ha", "median"),
    Median_Profit=("Profit_INR", "median"),
    Average_Water_Efficiency=("Water_Efficiency_t_per_1000m3", "mean")
).sort_values("Median_Yield", ascending=False)

irrigation_summary


In [ ]:
plt.figure(figsize=(9, 5))
plt.bar(
    irrigation_summary.index,
    irrigation_summary["Median_Yield"]
)
plt.title("Median Yield by Irrigation Method")
plt.xlabel("Irrigation Method")
plt.ylabel("Yield (tonnes/hectare)")
plt.xticks(rotation=15)
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()


### Irrigation Interpretation

Drip irrigation has the highest median yield among the irrigation methods in the supplied data.

However, a higher yield does not automatically mean the method is the most water-efficient. Yield and water efficiency should therefore be considered together.


## 13. Correlation Analysis

Correlation can help identify variables that move together. It does **not** prove causation.


In [ ]:
numeric_df = df.select_dtypes(include=np.number)

corr_with_profit = (
    numeric_df.corr()["Profit_INR"]
    .sort_values(ascending=False)
)

corr_with_profit


### Important Relationships with Profit

In [ ]:
profit_corr = corr_with_profit.drop("Profit_INR")

print("Strongest positive relationships with Profit:")
print(profit_corr.head(5))

print("\nStrongest negative relationships with Profit:")
print(profit_corr.tail(5))


## 14. Yield and Profit Relationship

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(
    df["Yield_Tonnes_Ha"],
    df["Profit_INR"],
    alpha=0.35
)
plt.title("Yield vs Profit")
plt.xlabel("Yield (tonnes/hectare)")
plt.ylabel("Profit (₹)")
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

yield_profit_corr = df["Yield_Tonnes_Ha"].corr(df["Profit_INR"])
print(f"Correlation between yield and profit: {yield_profit_corr:.3f}")


## 15. State-Level Comparison

The dataset covers multiple states. A state-level summary helps check whether seasonal patterns are spread across locations or concentrated in particular areas.


In [ ]:
state_summary = df.groupby("State").agg(
    Farms=("Farm_ID", "count"),
    Median_Yield=("Yield_Tonnes_Ha", "median"),
    Median_Profit=("Profit_INR", "median")
).sort_values("Median_Yield", ascending=False)

state_summary


## 16. Key Findings

Based on the analysis:

1. **Kharif is the strongest overall season** in terms of median yield and median profitability.
2. **Zaid has the weakest profitability**, with a negative median profit.
3. Seasonal environmental conditions differ substantially, especially rainfall and soil moisture.
4. **Drip irrigation has the highest median yield** among the irrigation methods in the dataset.
5. **Sugarcane is a standout crop** for yield and profitability.
6. Yield and profit show a positive relationship in the data, although correlation alone does not establish causation.
7. Water efficiency and profitability should be considered together instead of focusing only on production.


## 17. Recommendations

Based on the observed patterns, the following recommendations can be considered:

- Use seasonal performance comparisons when planning crop and resource allocation.
- Pay particular attention to Zaid-season cost and profitability conditions.
- Evaluate efficient irrigation methods alongside yield rather than looking at yield alone.
- Consider crop-specific seasonal performance when selecting crops.
- Combine agricultural data with local weather and market-price information for better decisions.
- Validate these observations with field-level data before using them for operational decisions.


## 18. Limitations

- The dataset is observational, so the analysis cannot establish cause-and-effect relationships.
- The dataset does not provide a long time series for studying year-to-year trends.
- Missing numeric values were imputed using medians.
- External factors such as detailed weather forecasts, market shocks and local farming practices are not modeled separately.
- The conclusions apply to the supplied dataset and should not automatically be generalized to all agricultural regions.


## 19. Conclusion

This analysis shows that **seasonal conditions are closely associated with differences in agricultural performance** in the supplied dataset.

Kharif shows the strongest overall combination of yield and profitability, while Zaid presents comparatively weaker economic outcomes. The analysis also shows useful differences across crops and irrigation methods.

Overall, the project demonstrates how simple data-cleaning, grouped analysis, correlation and visualization techniques can turn raw agricultural records into understandable insights.


## 20. Project Files

The GitHub repository is organized as:

```text
Seasonal-Agriculture-Performance-Analysis/
│
├── README.md
├── requirements.txt
├── data/
│   └── seasonal_agriculture_performance_dataset.csv
├── notebooks/
│   └── Seasonal_Agriculture_Performance_Analysis.ipynb
├── presentation/
│   └── Ranadeep_Das_Seasonal_Agriculture_Performance_Analysis_PPT.pptx
├── report/
│   └── Major_Project_Seasonal_Agriculture_Performance_Analysis.pdf
└── images/
```

**Student:** Ranadeep Das  
**AICTE STU ID:** `STU6a5f732ab94b61784640298`
